In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 19. 사전학습 — 언어 모델링의 수학 [CPU/GPU 벤치마크 ⑧]

> **학습 목표**
> - 인과적 언어 모델링(Causal LM)의 손실을 유도한다
> - 교사 강요(teacher forcing)의 의미를 이해한다
> - 학습 데이터 배치 구성 방법을 안다
> - 학습 루프 속도를 CPU vs GPU로 비교한다

## 19.1 사전학습의 목표 — 다음 토큰 예측

LLM 사전학습: 이전 토큰들로 다음 토큰을 예측.

$$\mathcal{L}_{\text{CLM}} = -\frac{1}{T} \sum_{t=1}^{T} \log P(w_t | w_{<t}; \theta)$$

- 인과적(causal): 미래 토큰 못 봄
- 교차 엔트로피 손실 (Ch 05)
- 자가 지도(self-supervised): 라벨 = 입력 시프트

## 19.2 교사 강요 (Teacher Forcing)

입력 $\mathbf{x} = [w_0, w_1, \ldots, w_{T-1}]$에 대해:
- 모델 입력: $[w_0, \ldots, w_{T-2}]$
- 정답: $[w_1, \ldots, w_{T-1}]$ (한 칸 shift)

한 번의 순전파로 $T-1$개 토큰 예측 학습 → 효율적.


In [ ]:

# 공통 모델 정의 (Ch 18과 동일)
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model 정의 완료")


In [ ]:
# Tiny Shakespeare로 사전학습
from llm_math.data import load_tiny_shakespeare
import numpy as np

text = load_tiny_shakespeare(max_chars=50000)
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = np.array([char_to_idx[c] for c in text])
print(f"텍스트 Length: {len(text):,}, Vocabulary Size: {vocab_size}")

# 배치 생성
def get_batch(data, seq_len, batch_size):
    starts = np.random.randint(0, len(data) - seq_len - 1, batch_size)
    x = np.stack([data[s:s+seq_len] for s in starts])
    y = np.stack([data[s+1:s+1+seq_len] for s in starts])  # shift
    return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# 학습
model = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Parameter Count: {n_params:,}")

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

n_steps = 300
losses = []
import time
t0 = time.time()
for step in range(n_steps):
    x, y = get_batch(data, seq_len=64, batch_size=32)
    opt.zero_grad()
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"스텝 {step+1}: loss={loss.item():.4f}")
print(f"\nTraining Time: {time.time() - t0:.1f}초")


In [ ]:
# 학습 곡선 및 PPL
import matplotlib.pyplot as plt

ppl = np.exp(losses)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(losses, alpha=0.3, color='blue')
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2)
axes[0].set_xlabel('스텝'); axes[0].set_ylabel('Loss')
axes[0].set_title('Pretraining Loss Curve')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ppl, alpha=0.3, color='green')
axes[1].plot(range(window-1, len(ppl)), np.exp(smoothed), 'r-', linewidth=2)
axes[1].set_xlabel('스텝'); axes[1].set_ylabel('PPL')
axes[1].set_title(f'Perplexity (최종: {np.exp(losses[-1]):.2f})')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch19_pretraining.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# 학습 후 텍스트 생성
def generate(model, prompt, n_new=200, temperature=0.8):
    model.eval()
    idx = [char_to_idx[c] for c in prompt]
    with torch.no_grad():
        for _ in range(n_new):
            x = torch.tensor([idx[-128:]], dtype=torch.long)  # 마지막 128
            logits = model(x)
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            idx.append(next_idx)
    return ''.join(idx_to_char[i] for i in idx)

print("=== Pretraining 후 Generation된 텍스트 ===\n")
print(generate(model, "ROMEO:\n", n_new=300, temperature=0.7))


## 19.3 [CPU/GPU 벤치마크 ⑧] 학습 루프 1 epoch 시간


In [ ]:
# 학습 속도 비교
import time
from llm_math.bench import time_fn

def train_one_step(model, x, y, opt, loss_fn):
    opt.zero_grad()
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    return loss

# CPU
model_cpu = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_cpu = torch.optim.AdamW(model_cpu.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()
x_cpu, y_cpu = get_batch(data, 64, 32)
res_cpu = time_fn(train_one_step, model_cpu, x_cpu, y_cpu, opt_cpu, loss_fn, device='cpu', warmup=2, repeat=5)
print(f"CPU 1 step: {res_cpu['mean_ms']:.3f} ms ({res_cpu['std_ms']:.3f} std)")

if torch.cuda.is_available():
    model_gpu = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128).cuda()
    opt_gpu = torch.optim.AdamW(model_gpu.parameters(), lr=3e-4)
    x_gpu, y_gpu = x_cpu.cuda(), y_cpu.cuda()
    res_gpu = time_fn(train_one_step, model_gpu, x_gpu, y_gpu, opt_gpu, loss_fn, device='cuda', warmup=3, repeat=10)
    print(f"GPU 1 step: {res_gpu['mean_ms']:.3f} ms ({res_gpu['std_ms']:.3f} std)")
    print(f"Speed Improvement: {res_cpu['mean_ms'] / res_gpu['mean_ms']:.2f}x")
else:
    print("GPU 없음. Model이 작아서 GPU 이득이 적을 수 있음.")

# Mixed Precision Effect (GPU만)
if torch.cuda.is_available():
    def train_step_amp(model, x, y, opt, loss_fn, scaler):
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        return loss
    scaler = torch.cuda.amp.GradScaler()
    res_amp = time_fn(train_step_amp, model_gpu, x_gpu, y_gpu, opt_gpu, loss_fn, scaler, device='cuda', warmup=3, repeat=10)
    print(f"GPU AMP 1 step: {res_amp['mean_ms']:.3f} ms (Mixed Precision)")


## 19.4 그래디언트 Accumulation

큰 배치가 필요하지만 GPU 메모리가 부족할 때: 작은 미니배치 여러 번 그래디언트 누적.

$$\theta \leftarrow \theta - \eta \frac{1}{K} \sum_{k=1}^{K} \nabla \mathcal{L}_k$$

효과적 배치 크기 = $K \times$ 미니배치 크기.


In [ ]:
# 그래디언트 accumulation 예시
def train_with_accumulation(model, data, opt, loss_fn, n_steps, accum_steps=4, batch_size=8):
    losses = []
    for step in range(n_steps):
        opt.zero_grad()
        total_loss = 0
        for _ in range(accum_steps):
            x, y = get_batch(data, 64, batch_size)
            logits = model(x)
            loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
            (loss / accum_steps).backward()  # 스케일링
            total_loss += loss.item()
        opt.step()
        losses.append(total_loss / accum_steps)
    return losses

# 비교: batch_size=32 (1회) vs batch_size=8 + accum=4 (같은 효과 배치)
torch.manual_seed(0)
model_a = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_a = torch.optim.AdamW(model_a.parameters(), lr=3e-4)
torch.manual_seed(0)
model_b = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
opt_b = torch.optim.AdamW(model_b.parameters(), lr=3e-4)

print("그래디언트 accumulation vs 일반 (같은 Effect Batch=32):")
# 일반
losses_normal = []
for step in range(30):
    opt_a.zero_grad()
    x, y = get_batch(data, 64, 32)
    logits = model_a(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    opt_a.step()
    losses_normal.append(loss.item())

# accumulation
losses_accum = train_with_accumulation(model_b, data, opt_b, loss_fn, 30, accum_steps=4, batch_size=8)

plt.figure(figsize=(10, 4))
plt.plot(losses_normal, label='batch=32 (1회)', alpha=0.7)
plt.plot(losses_accum, label='batch=8, accum=4 (누적)', alpha=0.7)
plt.xlabel('스텝'); plt.ylabel('Loss')
plt.title('그래디언트 Accumulation: 같은 Effect Batch Magnitude')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch19_grad_accum.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> 두 Method은 거의 동일한 Effect. 메모리 부족 시 accumulation 사용.")


## 19.5 Mixed Precision (FP16/BF16)

- FP32 대신 FP16/BF16으로 순전파/역전파 → 메모리 절반, 속도 2배
- 그래디언트는 FP32로 누적 (수치 안정성)
- `torch.cuda.amp.autocast()`로 자동 처리

CPU는 속도 향상 적지만, GPU에서 큰 이득.


## 19.6 핵심 정리

| 개념 | 수식/방식 | 의미 |
|---|---|---|
| Causal LM 손실 | $-\sum \log P(w_t\|w_{<t})$ | 다음 토큰 예측 |
| 교사 강요 | 입력 시프트 | 한 번에 T개 학습 |
| PPL | $\exp(\mathcal{L})$ | 모델 혼란도 |
| 그래디언트 accumulation | $\frac{1}{K}\sum \nabla\mathcal{L}$ | 메모리 절약 |
| Mixed precision | FP16 순전파, FP32 그래디언트 | 속도/메모리 |

## 연습문제

1. seq_len=32, 64, 128에 대해 학습 100스텝 후 PPL을 비교하라.
2. batch_size=8, 16, 32, 64로 학습하고 수렴 속도를 비교하라.
3. 그래디언트 accumulation으로 batch_size=64를 흉내 내라 (batch=16, accum=4).
4. Mixed precision이 GPU에서 주는 속도 향상을 측정하라.
5. 학습률 1e-4, 3e-4, 1e-3로 비교하고发散하는 경우를 관찰하라.

> 해설: `solutions/ch19_solutions.ipynb`
